# Cube Separator Defect Detector — V3 Training

## Stage 1 — Detection (EfficientNet B0)
"Is this photo showing a cube separator?"
→ NO  → NOISE (skip)
→ YES → Stage 2

## Stage 2 — Quality (EfficientNet B2 · 3-class · K-Fold Ensemble)
"Is this cube separator GOOD, BAD, or mid-service (CLEANING)?"
- Focal Loss (γ=2.0) · Label Smoothing (0.1)
- Mixup (α=0.4) · Progressive Resizing (160 → 224 px)
- 5-Fold CV with ensemble inference

## Drive folder structure
```
MyDrive/
  cube_separator_training/
    cube_separator_training.zip   ← zip containing good/ bad/ cleaning/ not_cube_separator/
    trained_models/               ← models saved here automatically
```

## Steps
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `cube_separator_training.zip` to Drive folder above
3. Run all cells top to bottom
4. Download model files at the end


In [ ]:
# Cell 1 — Imports
import pickle, shutil, random, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    efficientnet_b2, EfficientNet_B2_Weights,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

IMG_SIZE_SMALL = 160   # progressive resizing — warm-up
IMG_SIZE       = 224   # full resolution

def make_inference_tf(size=IMG_SIZE):
    return T.Compose([
        T.Resize(int(size * 256 / 224)),
        T.CenterCrop(size),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

INFERENCE_TF = make_inference_tf(IMG_SIZE)


In [ ]:
# Cell 2 — Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/cube_separator_training'
MODEL_OUT  = Path(DRIVE_BASE) / 'trained_models'
MODEL_OUT.mkdir(parents=True, exist_ok=True)

def load_images(folder: Path, label: str) -> list:
    if not folder.exists():
        raise FileNotFoundError(f'Folder not found: {folder}')
    paths = [p for p in folder.rglob('*')
             if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {label}: {len(paths)} photos')
    return paths

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


In [ ]:
# Cell 3 — Extract training zip from Drive to Colab local disk (fast I/O)
zip_path     = f'{DRIVE_BASE}/cube_separator_training.zip'
extract_path = '/content/'

print(f'Extracting {zip_path} ...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
print('Done.')

# Auto-detect the extracted folder name
LOCAL_BASE = Path('/content/cube_separator_training')
if not LOCAL_BASE.exists():
    # zip might not have a top-level folder — photos directly at /content/
    LOCAL_BASE = Path('/content')

GOOD_DIR   = LOCAL_BASE / 'good'
BAD_DIR    = LOCAL_BASE / 'bad'
CLEAN_DIR  = LOCAL_BASE / 'cleaning'
NOT_CS_DIR = LOCAL_BASE / 'not_cube_separator'

print(f'\nLoading from: {LOCAL_BASE}\n')
good_paths    = load_images(GOOD_DIR,   'good     (normal install)')
bad_paths     = load_images(BAD_DIR,    'bad      (defect)')
clean_paths   = load_images(CLEAN_DIR,  'cleaning (mid-service)')
not_cs_paths  = load_images(NOT_CS_DIR, 'not_cube_separator (noise)')

if len(good_paths) < 10:
    raise RuntimeError(f'Only {len(good_paths)} good photos — need at least 10')
if len(bad_paths) < 10:
    raise RuntimeError(f'Only {len(bad_paths)} bad photos — need at least 10')
if len(not_cs_paths) < 10:
    raise RuntimeError(f'Only {len(not_cs_paths)} noise photos — add more')

print(f'\nStage 1 positives : {len(good_paths) + len(bad_paths) + len(clean_paths)}')
print(f'Stage 1 negatives : {len(not_cs_paths)}')
print(f'Stage 2 classes   : good={len(good_paths)}  bad={len(bad_paths)}  cleaning={len(clean_paths)}')


In [ ]:
# Cell 4 — Build Stage 1 detection dataset
# Positive: good + bad + cleaning (all ARE cube separator photos)
# Negative: not_cube_separator
DETECT_ROOT = Path('cs_detect_split')
for split in ('train', 'val'):
    for cls in ('cube_separator', 'not_cube_separator'):
        (DETECT_ROOT / split / cls).mkdir(parents=True, exist_ok=True)

random.seed(42)

pos_paths = good_paths + bad_paths + clean_paths
random.shuffle(pos_paths)
split_pos = max(1, int(len(pos_paths) * 0.85))
for p in pos_paths[:split_pos]:
    shutil.copy(p, DETECT_ROOT / 'train' / 'cube_separator' / p.name)
for p in pos_paths[split_pos:]:
    shutil.copy(p, DETECT_ROOT / 'val' / 'cube_separator' / p.name)

neg_paths = list(not_cs_paths)
random.shuffle(neg_paths)
split_neg = max(1, int(len(neg_paths) * 0.85))
for p in neg_paths[:split_neg]:
    shutil.copy(p, DETECT_ROOT / 'train' / 'not_cube_separator' / p.name)
for p in neg_paths[split_neg:]:
    shutil.copy(p, DETECT_ROOT / 'val' / 'not_cube_separator' / p.name)

for split in ('train', 'val'):
    counts = {c: len(list((DETECT_ROOT/split/c).iterdir()))
              for c in ('cube_separator', 'not_cube_separator')}
    print(f'{split}: {counts}')


In [ ]:
# Cell 5 — Train Stage 1 detector (EfficientNet B0 — binary)
BATCH = 16

det_train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

det_train_ds = ImageFolder(str(DETECT_ROOT / 'train'), transform=det_train_tf,
                           is_valid_file=is_valid_image)
det_val_ds   = ImageFolder(str(DETECT_ROOT / 'val'),   transform=INFERENCE_TF,
                           is_valid_file=is_valid_image)

counts_det = Counter(det_train_ds.targets)
weights    = [1.0 / counts_det[t] for t in det_train_ds.targets]
sampler    = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

det_train_loader = DataLoader(det_train_ds, batch_size=BATCH, sampler=sampler,  num_workers=2)
det_val_loader   = DataLoader(det_val_ds,   batch_size=BATCH, shuffle=False,    num_workers=2)

print('class_to_idx:', det_train_ds.class_to_idx)
CS_IDX = det_train_ds.class_to_idx['cube_separator']
print(f'cube_separator class index: {CS_IDX}')

# ── Model builders ────────────────────────────────────────────────────────────
def make_b0(num_classes: int):
    net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    for p in net.parameters(): p.requires_grad = False
    net.classifier = nn.Sequential(
        nn.Dropout(p=0.3), nn.Linear(net.classifier[1].in_features, num_classes)
    )
    return net.to(DEVICE)

def make_b2(num_classes: int):
    net = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)
    for p in net.parameters(): p.requires_grad = False
    net.classifier = nn.Sequential(
        nn.Dropout(p=0.3), nn.Linear(net.classifier[1].in_features, num_classes)
    )
    return net.to(DEVICE)

def train_simple(model, train_loader, val_loader, epochs, lr, unfreeze_blocks=None):
    """Simple training loop for Stage 1 (no Mixup, no Focal Loss)."""
    if unfreeze_blocks:
        for name, param in model.named_parameters():
            if any(b in name for b in unfreeze_blocks) or 'classifier' in name:
                param.requires_grad = True
    params    = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.Adam(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_acc, best_state = 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        tr_correct, tr_total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tr_correct += (out.argmax(1) == labels).sum().item()
            tr_total   += len(imgs)
        model.eval()
        vl_correct, vl_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                vl_correct += (model(imgs).argmax(1) == labels).sum().item()
                vl_total   += len(imgs)
        vl_acc = vl_correct / max(1, vl_total)
        scheduler.step()
        flag = ''
        if vl_acc > best_acc:
            best_acc   = vl_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            flag = '  ← saved'
        print(f'Ep {epoch:2d} | train={tr_correct/max(1,tr_total):.3f} | val={vl_acc:.3f}{flag}')
    model.load_state_dict(best_state)
    return best_acc

print('\nBuilding Stage 1 detector (EfficientNet B0, binary)...')
detector = make_b0(num_classes=2)

print('Phase 1 — head only (12 epochs)')
acc1 = train_simple(detector, det_train_loader, det_val_loader, epochs=12, lr=1e-3)

print('\nPhase 2 — fine-tune last blocks (12 epochs)')
acc2 = train_simple(detector, det_train_loader, det_val_loader, epochs=12, lr=1e-4,
                    unfreeze_blocks=['features.7', 'features.8'])

print(f'\nBest Stage 1 val acc: {max(acc1, acc2):.3f}')


In [ ]:
# Cell 6 — Evaluate and save Stage 1 detector
detector.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in det_val_loader:
        preds = detector(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=det_val_ds.classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=det_val_ds.classes, yticklabels=det_val_ds.classes, cmap='Greens')
plt.title('Stage 1 — Cube Separator Detector Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('cs_detector_confusion.png', dpi=150)
plt.show()

det_bundle = {
    'model_state_dict': detector.state_dict(),
    'class_to_idx':     det_train_ds.class_to_idx,
    'idx_to_class':     {v: k for k, v in det_train_ds.class_to_idx.items()},
    'cs_class_idx':     CS_IDX,
    'architecture':     'efficientnet_b0',
    'input_size':       IMG_SIZE,
    'val_acc':          round(max(acc1, acc2), 4),
    'version':          3,
}
torch.save(det_bundle, 'cube_separator_detector.pt')
shutil.copy('cube_separator_detector.pt', MODEL_OUT / 'cube_separator_detector.pt')
print(f'Saved: cube_separator_detector.pt  (val_acc={det_bundle["val_acc"]})')


In [ ]:
# Cell 7 — Stage 2 Quality Model
# EfficientNet B2 · Focal Loss (γ=2.0) · Mixup (α=0.4)
# Progressive Resizing (160→224px) · 5-Fold CV · WeightedRandomSampler

# ── Class setup ───────────────────────────────────────────────────────────────
CLASS_TO_IDX = {'bad': 0, 'cleaning': 1, 'good': 2}   # alphabetical
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES  = 3
BAD_IDX      = CLASS_TO_IDX['bad']

# ── Custom dataset ────────────────────────────────────────────────────────────
class FileListDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs     = [(str(p), l) for p, l in pairs]
        self.transform = transform
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        return self.transform(Image.open(path).convert('RGB')), label

# ── Focal Loss with label smoothing ──────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.1, num_classes=NUM_CLASSES):
        super().__init__()
        self.gamma = gamma
        self.ls    = label_smoothing
        self.nc    = num_classes
    def forward(self, logits, targets):
        with torch.no_grad():
            oh = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
            oh = oh * (1 - self.ls) + self.ls / self.nc
        log_p  = F.log_softmax(logits, dim=1)
        p_t    = (oh * log_p.exp()).sum(1)
        focal  = (1 - p_t) ** self.gamma
        return (focal * -(oh * log_p).sum(1)).mean()

# ── Mixup ─────────────────────────────────────────────────────────────────────
def mixup_batch(imgs, labels, alpha=0.4):
    lam    = float(np.random.beta(alpha, alpha))
    idx    = torch.randperm(imgs.size(0), device=imgs.device)
    mixed  = lam * imgs + (1 - lam) * imgs[idx]
    oh     = F.one_hot(labels, NUM_CLASSES).float()
    soft_y = lam * oh + (1 - lam) * oh[idx]
    return mixed, soft_y

def focal_soft(logits, soft_y, gamma=2.0):
    """Focal loss for Mixup soft labels."""
    log_p = F.log_softmax(logits, dim=1)
    p_t   = (soft_y * log_p.exp()).sum(1)
    focal = (1 - p_t) ** gamma
    return (focal * -(soft_y * log_p).sum(1)).mean()

# ── Train one fold ────────────────────────────────────────────────────────────
def train_fold(model, train_loader, val_loader, use_mixup=True):
    focal_loss = FocalLoss(gamma=2.0, label_smoothing=0.1)
    best_acc, best_state = 0.0, None

    def _run_phase(loader_size, loader_280, epochs, lr, unfreeze=None):
        nonlocal best_acc, best_state
        if unfreeze:
            for name, param in model.named_parameters():
                if any(b in name for b in unfreeze) or 'classifier' in name:
                    param.requires_grad = True
        params    = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = optim.Adam(params, lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        for epoch in range(1, epochs + 1):
            model.train()
            tr_correct, tr_total = 0, 0
            for imgs, labels in loader_size:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                if use_mixup:
                    mixed, soft_y = mixup_batch(imgs, labels)
                    out  = model(mixed)
                    loss = focal_soft(out, soft_y)
                else:
                    out  = model(imgs)
                    loss = focal_loss(out, labels)
                loss.backward()
                optimizer.step()
                tr_correct += (out.argmax(1) == labels).sum().item()
                tr_total   += len(imgs)
            model.eval()
            vl_correct, vl_total = 0, 0
            with torch.no_grad():
                for imgs, labels in loader_280:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    vl_correct += (model(imgs).argmax(1) == labels).sum().item()
                    vl_total   += len(imgs)
            vl_acc = vl_correct / max(1, vl_total)
            scheduler.step()
            flag = ''
            if vl_acc > best_acc:
                best_acc   = vl_acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                flag = '  ← saved'
            print(f'    Ep {epoch:2d} | train={tr_correct/max(1,tr_total):.3f} | val={vl_acc:.3f}{flag}')

    # Progressive resizing: warm-up at 160px, then full 224px
    print('  Phase 1 — 160px  head only  (8 epochs)')
    _run_phase(train_loader['small'], val_loader, epochs=8,  lr=1e-3)
    print('  Phase 2 — 224px  head only  (8 epochs)')
    _run_phase(train_loader['full'],  val_loader, epochs=8,  lr=5e-4)
    print('  Phase 3 — 224px  fine-tune  (12 epochs)')
    _run_phase(train_loader['full'],  val_loader, epochs=12, lr=5e-5,
               unfreeze=['features.6', 'features.7', 'features.8'])

    model.load_state_dict(best_state)
    return best_acc

# ── Transforms ────────────────────────────────────────────────────────────────
def make_aug_tf(size):
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.65, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
        T.RandomRotation(15),
        T.RandomPerspective(distortion_scale=0.25, p=0.4),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

aug_tf_small = make_aug_tf(IMG_SIZE_SMALL)
aug_tf_full  = make_aug_tf(IMG_SIZE)

# ── Build orig pairs + K-Fold ─────────────────────────────────────────────────
orig_pairs = (
    [(p, CLASS_TO_IDX['good'])     for p in good_paths] +
    [(p, CLASS_TO_IDX['bad'])      for p in bad_paths]  +
    [(p, CLASS_TO_IDX['cleaning']) for p in clean_paths]
)
orig_labels = [l for _, l in orig_pairs]

N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

fold_state_dicts = []
fold_val_accs    = []

for fold, (train_idx, val_idx) in enumerate(skf.split(orig_pairs, orig_labels)):
    print(f'\n{"="*55}')
    print(f'FOLD {fold+1}/{N_FOLDS}')

    train_pairs = [orig_pairs[i] for i in train_idx]
    val_pairs   = [orig_pairs[i] for i in val_idx]

    tr_counts = defaultdict(int)
    vl_counts = defaultdict(int)
    for _, l in train_pairs: tr_counts[IDX_TO_CLASS[l]] += 1
    for _, l in val_pairs:   vl_counts[IDX_TO_CLASS[l]] += 1
    print(f'  Train: {dict(tr_counts)}')
    print(f'  Val  : {dict(vl_counts)}')

    # Weighted sampler for train class balance
    tr_lbls  = [l for _, l in train_pairs]
    counts_f = Counter(tr_lbls)
    wts      = [1.0 / counts_f[l] for l in tr_lbls]
    sampler_f = WeightedRandomSampler(wts, num_samples=len(wts), replacement=True)

    train_ds_small = FileListDataset(train_pairs, aug_tf_small)
    train_ds_full  = FileListDataset(train_pairs, aug_tf_full)
    val_ds         = FileListDataset(val_pairs,   INFERENCE_TF)

    # Separate samplers needed per loader (sampler state is shared — recreate)
    def make_sampler(pairs):
        lbls = [l for _, l in pairs]
        c    = Counter(lbls)
        w    = [1.0 / c[l] for l in lbls]
        return WeightedRandomSampler(w, num_samples=len(w), replacement=True)

    tr_loader = {
        'small': DataLoader(train_ds_small, batch_size=16, sampler=make_sampler(train_pairs), num_workers=2),
        'full':  DataLoader(train_ds_full,  batch_size=16, sampler=make_sampler(train_pairs), num_workers=2),
    }
    vl_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

    model = make_b2(num_classes=NUM_CLASSES)
    best  = train_fold(model, tr_loader, vl_loader, use_mixup=True)

    fold_state_dicts.append({k: v.clone() for k, v in model.state_dict().items()})
    fold_val_accs.append(best)
    print(f'  Fold {fold+1} best val acc: {best:.3f}')

print(f'\n{"="*55}')
print('K-Fold complete:')
for i, acc in enumerate(fold_val_accs):
    print(f'  Fold {i+1}: {acc:.3f}')
print(f'Mean : {np.mean(fold_val_accs):.3f}  ±  {np.std(fold_val_accs):.3f}')


In [ ]:
# Cell 8 — Calibrate confidence threshold
# For 3-class, predicted label = argmax(softmax).
# If max(softmax) < MIN_CONFIDENCE → UNCERTAIN (not written as bad/good/cleaning).
# We score all originals with the ensemble to pick a safe MIN_CONFIDENCE.

ensemble_nets = []
for sd in fold_state_dicts:
    net = make_b2(num_classes=NUM_CLASSES)
    net.load_state_dict(sd)
    net.eval().to(DEVICE)
    ensemble_nets.append(net)

all_orig_ds     = FileListDataset(orig_pairs, INFERENCE_TF)
all_orig_loader = DataLoader(all_orig_ds, batch_size=16, shuffle=False, num_workers=2)

confidences_by_class = defaultdict(list)   # true_class → list of max(softmax)
correct_by_class     = defaultdict(list)   # true_class → list of bool (correct?)

with torch.no_grad():
    for imgs, labels in all_orig_loader:
        imgs = imgs.to(DEVICE)
        probs_avg = sum(
            torch.softmax(net(imgs), dim=1) for net in ensemble_nets
        ) / N_FOLDS
        confidence, pred = probs_avg.max(1)
        for conf, p, true in zip(confidence.cpu().numpy(),
                                  pred.cpu().numpy(),
                                  labels.numpy()):
            cls_name = IDX_TO_CLASS[true]
            confidences_by_class[cls_name].append(float(conf))
            correct_by_class[cls_name].append(int(p) == int(true))

print('Ensemble confidence stats (on all originals):')
for cls in ('good', 'bad', 'cleaning'):
    confs   = confidences_by_class[cls]
    correct = correct_by_class[cls]
    acc     = sum(correct) / len(correct) if correct else 0
    print(f'  {cls:10s}  n={len(confs):4d}  acc={acc:.3f}  '
          f'conf_median={np.median(confs):.3f}  conf_min={min(confs):.3f}')

# Min confidence = 5th percentile of all correct predictions
all_correct_confs = [
    c for cls in ('good', 'bad', 'cleaning')
    for c, ok in zip(confidences_by_class[cls], correct_by_class[cls]) if ok
]
MIN_CONFIDENCE = round(float(np.percentile(all_correct_confs, 5)), 4)
print(f'\nCalibrated MIN_CONFIDENCE : {MIN_CONFIDENCE:.4f}')
print(f'  (5th percentile of correct-prediction confidences)')
print(f'  Photos below this → UNCERTAIN in the report')

# Distribution plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {'good': 'green', 'bad': 'red', 'cleaning': 'orange'}
for ax, cls in zip(axes, ('good', 'bad', 'cleaning')):
    ax.hist(confidences_by_class[cls], bins=20, color=colors[cls], alpha=0.75)
    ax.axvline(MIN_CONFIDENCE, color='black', linestyle='--', linewidth=1.5,
               label=f'threshold={MIN_CONFIDENCE:.3f}')
    ax.set_title(f'{cls}  (n={len(confidences_by_class[cls])})')
    ax.set_xlabel('Max softmax confidence')
    ax.legend(fontsize=8)
plt.suptitle('Stage 2 Ensemble — Confidence Distributions')
plt.tight_layout()
plt.savefig('cs_threshold.png', dpi=150)
plt.show()


In [ ]:
# Cell 9 — Save models and download
quality_bundle = {
    'fold_state_dicts': fold_state_dicts,
    'class_to_idx':     CLASS_TO_IDX,
    'idx_to_class':     IDX_TO_CLASS,
    'bad_class_idx':    BAD_IDX,
    'architecture':     'efficientnet_b2',
    'input_size':       IMG_SIZE,
    'threshold':        MIN_CONFIDENCE,
    'good_count':       len(good_paths),
    'bad_count':        len(bad_paths),
    'cleaning_count':   len(clean_paths),
    'k_folds':          N_FOLDS,
    'fold_val_accs':    fold_val_accs,
    'mean_val_acc':     round(float(np.mean(fold_val_accs)), 4),
    'version':          3,
}
torch.save(quality_bundle, 'cube_separator_quality.pt')
shutil.copy('cube_separator_quality.pt', MODEL_OUT / 'cube_separator_quality.pt')
shutil.copy('cs_threshold.png',          MODEL_OUT / 'cs_threshold.png')

print('Stage 2 [B2 ensemble 5-fold] saved: cube_separator_quality.pt')
print(f'  good_count    : {quality_bundle["good_count"]}')
print(f'  bad_count     : {quality_bundle["bad_count"]}')
print(f'  cleaning_count: {quality_bundle["cleaning_count"]}')
print(f'  mean_val_acc  : {quality_bundle["mean_val_acc"]}')
print(f'  fold_val_accs : {[round(a, 3) for a in fold_val_accs]}')
print(f'  threshold     : {quality_bundle["threshold"]:.4f}  (min_confidence)')
print(f'  architecture  : efficientnet_b2')

print('\n' + '='*60)
print('Downloading models...')
from google.colab import files
files.download('cube_separator_detector.pt')
files.download('cube_separator_quality.pt')
files.download('cs_detector_confusion.png')
files.download('cs_threshold.png')

print('\nNext steps:')
print('  1. Drop cube_separator_detector.pt into  cube_separator_detector-Final/')
print('  2. Drop cube_separator_quality.pt  into  cube_separator_detector-Final/')
print('  3. Run: python3 run_cube_separator_report.py')
